In [ ]:
%matplotlib inline

In [ ]:
from essential.kegg_modules import list_kegg_modules, kegg_module_to_graph, metabolic_to_operational_graph
from essential.plot_pathways import plot_pathway_results, plot_metabolic_pathway
from essential.utils import PLOTNINE_DEFAULT_THEME_2
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotnine as gg
import os
from tqdm import tqdm
import scanpy as sc
from essential.pathway_discontinuity import PathwayDiscontinuity


plt.rcParams["svg.fonttype"] = "none"

def plot_umap_genes(adata, genes):
    obs_subset = adata.obs.loc[lambda x: x["target"].isin(genes)]
    obs_subset["target"] = obs_subset["target"].astype(str)

    fig = (
        gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
        + gg.geom_point()
        + gg.geom_point(obs_subset, gg.aes(color="target"), size=2)
        + gg.theme_minimal()
    )
    return fig

def plot_umap_equiv_classes(adata, gene_to_class, class_color_mapping, plot_legend=True, point_size=1.5):
    obs_subset = adata.obs.loc[lambda x: x["target"].isin(gene_to_class.keys())]
    obs_subset["equivalence_class"] = obs_subset["target"].map(gene_to_class).sample(frac=1)

    fig = (
        gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
        + gg.geom_point(size=0.7, stroke=0)
        + gg.geom_point(obs_subset, gg.aes(color="equivalence_class"), size=point_size, stroke=0.0)
        + gg.scale_color_manual(values=class_color_mapping)
        + gg.theme_minimal()
    )
    if not plot_legend:
        fig = fig + gg.theme(legend_position="none")
    return fig

#### Import transcriptomic data

In [ ]:
!ls ../../../..

In [ ]:
from essential.data import load_fitness_data

fitness_data_path = "../../data/calvo2020_dcas9fitness/Supp_data2_log2FC.csv"
# fitness_data_path = "../../../../data/calvo2020_dcas9fitness/Supp_data2_log2FC.csv"
fitness_data = load_fitness_data(fitness_data_path).groupby("gene")[["T1", "T2", "T3", "T4"]].mean().reset_index()

In [ ]:
adata = sc.read_h5ad("../../data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.h5ad")
# adata = sc.read_h5ad("../../../../data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.h5ad")


sc.pp.neighbors(adata, n_neighbors=10, use_rep="X_pca")
sc.tl.umap(adata, min_dist=0.5)

adata.obs["UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["UMAP2"] = adata.obsm["X_umap"][:, 1]

mapping = {
   "0": "WT-like baseline",
   "1": "WT-like baseline",
   "2": "WT-like baseline",
   "3": "WT-like baseline",
   "4": "WT-like baseline",
   "5": "Mixed metabolism/respiration",
   "6": "Growth arrest",
   "7": "Envelope stress",
   "8": "Translation",
   "9": "WT-like baseline",
   "10": "Phosphate starvation"
}

adata.obs["leiden_label"] = adata.obs["leiden"].map(mapping)
# sc.pl.umap(adata, color="leiden_label")


import matplotlib as mpl
import plotnine as gg
labels = adata.obs["leiden_label"].astype("category")
tab10 = [mpl.colors.to_hex(c) for c in mpl.cm.get_cmap("tab10").colors]
color_map = {lab: tab10[i % len(tab10)] for i, lab in enumerate(labels.cat.categories)}

fig = (
   gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2", color="leiden_label"))
   + gg.geom_point(size=0.5, stroke=0)
   + gg.scale_color_manual(values=color_map)
   + gg.guides(color=gg.guide_legend(override_aes={"size": 3, "alpha": 1}))
   + gg.theme_minimal()
   + PLOTNINE_DEFAULT_THEME_2
   + gg.theme(
      figure_size=(3.5, 2)
   )
   + gg.labs(
      color=""
   )
)
fig.save("figures/leiden_labels.png", dpi=1000)
fig

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden_label", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20, show=False)
# plt.savefig("figures/leiden_label_rank_genes.png", dpi=1000)
# plt.close()


#### Mine KEGG modules

In [ ]:
module_info = pd.DataFrame(list_kegg_modules("eco"))
for i, row in tqdm(module_info.iterrows()):
    module_name = row["module_id"]
    g = kegg_module_to_graph(module_name, 'eco')
    genes = np.unique([d['gene'] for u, v, d in g.edges(data=True) if d['gene']])
    np.sort(genes)
    genes_str = ', '.join(genes)

    op = metabolic_to_operational_graph(g)

    pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
    results = pda.fit(threshold=0.1, mode="mmd_stat")

    module_info.loc[i, "n_equivalences"] = results.n_equivalences
    module_info.loc[i, "module_has_surprises"] = results.n_equivalences >= 2
    module_info.loc[i, "genes"] = genes_str
    module_info.loc[i, "n_genes"] = len(genes)
module_info.to_csv("module_info.csv", index=False)
module_info.to_json("module_info.json", orient="records", indent=2)


module_prediction = pd.read_json("module_prediction.json")
module_info_ = module_info.merge(module_prediction, on="module_id", how="left").query("n_genes >= 2")

fig = (
    gg.ggplot(
        module_info_,
        gg.aes(x="factor(activity_prediction)", fill="factor(module_has_surprises)")
    ) 
    + gg.geom_bar(position="dodge", width=0.5) 
    + gg.labs(
        x="activity prediction",
        y="# of modules",
        fill="module has surprises"
    ) 
    + gg.theme_minimal()
    + gg.scale_y_continuous(expand=(0, 0))
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        figure_size=(2.7, 2)
    )
)
fig.save("figures/module_has_surprises_by_activity_prediction.svg")
fig

In [ ]:
DISPLAY_TOP_K_MODULES = 10

display(module_info_.query("activity_prediction == 'active'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

display(module_info_.query("activity_prediction == 'partially active'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

display(module_info_.query("activity_prediction == 'inactive'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

# Batched experiment

In [ ]:
# EXAMPLES = [
#     "M00938",  # dUTP toxicity,
#     "M00120", # "coA biosynthesis"
#     "M00063",  # CMP-KDO biosynthesis
#     "M00121", # "heme
# ]

In [ ]:
selected_modules = module_info_.query("activity_prediction == 'active'").query("module_has_surprises").sort_values("n_equivalences", ascending=False)

for _, row in selected_modules.iterrows():
    module_id = row["module_id"]
    print(module_id)

    g = kegg_module_to_graph(module_id, 'eco', print_info=False)
    op = metabolic_to_operational_graph(g)

    pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
    results = pda.fit(threshold=0.2, mode="mmd_stat")

    fig, ax, plot_info = plot_pathway_results(
        g, 
        results.edge_equivalence, 
        x_sep=0.5, 
        y_sep=2, 
        label_offset=(0, 0.4), 
        edge_width=1.7, 
        color_surprising_individually=True
    )
    # display(fig)
    plt.savefig(f"figures/eco_{module_id}.svg")

    fig = plot_umap_equiv_classes(
        adata, 
        plot_info["gene_to_class"], 
        plot_info["class_color_mapping"], 
        plot_legend=False,
        point_size=5.0,
    ) + PLOTNINE_DEFAULT_THEME_2
    # display(fig)
    fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)


# Short-listed modules

In [ ]:
# Export pathway metadata
PATH_TO_MODULE_CARDS = "/Users/pboyeau/Projects/essential/experiments/04152026_kegg/pathway_analysis/short_listed_pathways_active"
os.makedirs(PATH_TO_MODULE_CARDS, exist_ok=True)

modules_of_interest = ["eco_M00001", 
"eco_M00009", 
"eco_M00011", 
"eco_M00049", 
"eco_M00115", 
"eco_M00120", 
"eco_M00125", 
"eco_M00364", 
"eco_M00121", 
"eco_M00120", ]

for module_id in modules_of_interest:
    print(module_id)
    g = kegg_module_to_graph(module_id, 'eco', print_info=True)
    str_ = g.graph["info_str"]
    filename = f"{PATH_TO_MODULE_CARDS}/{module_id}.md"
    str_ = "**observations**\n" + str_ + "\n"
    with open(filename, "w") as f:
        f.write(str_)
    

In [ ]:
def plot_expression_data(adata, perturbations, gene_names):
    adata_selected = adata[adata.obs["target"].isin(perturbations)]
    sc.pl.heatmap(adata_selected, var_names=gene_names, groupby="target", swap_axes=False)
    sc.pl.dotplot(adata_selected, var_names=gene_names, groupby="target", swap_axes=False)

In [ ]:
def compare_pairs(adata, gene1, gene2):
    adata_pair = adata[adata.obs["target"].isin([gene1, gene2])]
    sc.tl.rank_genes_groups(adata_pair, "target", method="wilcoxon")
    sc.pl.rank_genes_groups(adata_pair, n_genes=20, show=False)
    de_genes = sc.get.rank_genes_groups_df(adata_pair, group=gene1)

    for _, row in de_genes.sort_values("pvals").head(20).iterrows():
        print(f"{row['names']} pval:{row['pvals_adj']:0.2e}, log2FC:{row['logfoldchanges']:0.2f}")
    return de_genes

def rename_nodes(g, renamer):
    G_copy = g.copy()
    for node, new_name in renamer.items():
        if node in G_copy.nodes:
            G_copy.nodes[node]["name"] = new_name
    return G_copy


In [ ]:
results.edge_equivalence

In [ ]:
%matplotlib inline

#### Glycolysis

In [ ]:
module_id = "eco_M00001"
# module_id = "eco_M00049"

SAVE_DIR = "figures/figures_glycolysis"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
g = kegg_module_to_graph(module_id, 'eco', print_info=False)
for node, data in g.nodes(data=True):
    print(node, data.get("name"))

In [ ]:
op_u = op.to_undirected()

labels = {node: data.get("name", node) for node, data in op.nodes(data=True)}
fig, ax = plt.subplots(figsize=(4, 4))
nx.draw(op_u, labels=labels, with_labels=True, ax=ax,
        node_size=500, font_size=8, arrows=True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/operational_graph.svg")

In [ ]:
renamer = {
   "C00267": "Glc",
   "C00668": "G6P",
   "C00085": "F6P",
   "C00354": "FBP",
   "C00111": "DHAP",
   "C00118": "GAP",
   "C00236": "1,3-BPG",
   "C00197": "3PG",
   "C00631": "2PG",
   "C00074": "PEP",
   "C00022": "Pyr"
}
gp = rename_nodes(g, renamer)



In [ ]:
# g = kegg_module_to_graph(module_id, 'eco', print_info=False)
for node, data in gp.nodes(data=True):
    print(node, data.get("name"))

In [ ]:
op = metabolic_to_operational_graph(gp)
pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
# results = pda.fit(threshold=0.2, mode="mmd_stat")
results = pda.fit(threshold=-np.log(0.05), mode="mmd_pvalue")

In [ ]:
fig, ax, plot_info = plot_pathway_results(
    gp, 
    results.edge_equivalence, 
    x_sep=0.4, 
    y_sep=2, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True,
    show_legend=False,
    # node_mode="none"
    node_mode="labels",
    isoenzyme_spacing=0.05
)
# display(fig)
plt.savefig(f"{SAVE_DIR}/metabolic_graph.svg")

fig = plot_umap_equiv_classes(
    adata, 
    plot_info["gene_to_class"], 
    plot_info["class_color_mapping"], 
    plot_legend=False,
    point_size=2.0,
) + PLOTNINE_DEFAULT_THEME_2
display(fig)
fig.save(f"{SAVE_DIR}/umap.png", dpi=500)

In [ ]:
module_id = "eco_M00001"

perturbations = [
    "glk", "pgi", "pfkA", "pfkB", 
    "fbaB",
    "tpiA",
    "fbaA",  "gapA", "pgk", "eno", "pykF"
]
# gene_names = [
#     "edd", "eda", "zwf", "gnd", "mgsA", 
#     "gloA", "gloB"
# ]

gene_names = [
    "uspA",  # energy/stress-responsive; senses ATP directly as cofactor
    "rpoS",  # it implies a SHIFT in transcriptional priority toward stress-protective genes
    "relA",  # RelA synthesizes ppGpp → triggers stringent response (slows down translation)
    "spoT",  # bifunctional ppGpp synthetase/hydrolase; primary responder to non-amino-acid stresses
    "alaS",  # aminoacyl-tRNA synthetase for alanine; proxy for translation fidelity / amino acid availability
    "amn",   # AMP nucleosidase; degrades AMP → adenine + R5P; marker of AMP accumulation
    "purR"   # transcriptional repressor of de novo purine synthesis; de-repressed when purines are limiting
]

plot_expression_data(adata, perturbations, gene_names)

In [ ]:
import pandas as pd
import scanpy as sc


def plot_expression_data2(
    adata,
    perturbation_groups,
    gene_groups,
    layer=None,
    standard_scale="var",
    figsize=(8, 6),
):
    """Heatmap + dotplot with grouped columns (gene function) and ordered
    rows (perturbation phenotype class)."""
    perturbations = [p for _, ps in perturbation_groups for p in ps]
    gene_names = [g for _, gs in gene_groups for g in gs]

    adata_sel = adata[adata.obs["target"].isin(perturbations)].copy()
    adata_sel.obs["target"] = pd.Categorical(
        adata_sel.obs["target"].astype(str),
        categories=perturbations,
        ordered=True,
    )

    def group_positions(groups):
        pos, labels, i = [], [], 0
        for label, items in groups:
            pos.append((i, i + len(items) - 1))
            labels.append(label)
            i += len(items)
        return pos, labels

    var_pos, var_labels = group_positions(gene_groups)

    common = dict(
        var_names=gene_names,
        groupby="target",
        var_group_positions=var_pos,
        var_group_labels=var_labels,
        var_group_rotation=0,
        swap_axes=False,
        layer=layer,
        figsize=figsize,
    )

    # sc.pl.heatmap(adata_sel, standard_scale=standard_scale, **common)
    sc.pl.dotplot(adata_sel, standard_scale=standard_scale, **common)


module_id = "eco_M00001"

perturbation_groups = [
    ("non-essential (upper)", ["glk", "pgi", "pfkA", "pfkB"]),
    ("isozyme / bypassable", ["fbaB", "tpiA"]),
    ("essential trunk",      ["fbaA", "gapA", "pgk", "eno"]),
    ("PEP->pyr",             ["pykF"]),
]

# gene_groups = [
#     ("stress",       ["uspA", "rpoS"]),
#     ("stringent",    ["relA", "spoT"]),
#     ("translation",  ["alaS"]),
#     ("purine pool",  ["amn", "purR"]),
# ]
gene_groups = [
    ("Cra / gluconeogenesis",  ["fbaB", "ppsA", "fbp", "aceA", "aceB", "maeB"]),
    ("stringent / RpoS",       ["spoT", "relA", "rpoS", "rmf", "dps",
                                "tktB", "talA",          # RpoS-dependent PPP
                                "tktA", "zwf"]),          # σ70 PPP (expected down)
    # ("discriminators (tpiA)",         ["mgsA", "gloA", "gloB"]),
    # ("discriminators (eno)",          ["ptsG", "rne", "pnp"]),  # eno-specific (degradosome)
]

plot_expression_data2(adata, perturbation_groups, gene_groups, figsize=(12, 5))

In [ ]:
adata_sel = adata[adata.obs["target"].isin(perturbations)].copy()
sc.pp.pca(adata_sel, n_comps=10)
sc.pp.neighbors(adata_sel, use_rep="X_pca")
sc.tl.umap(adata_sel)
sc.pl.umap(adata_sel, color=["target"], )

In [ ]:
fitness_data.loc[lambda x: x["gene"].isin(perturbations)]

## Pathway-level assessment

**pathway_description:** The Embden-Meyerhof-Parnas pathway converts glucose to pyruvate, producing ATP, NADH, and biosynthetic precursors (G6P, F6P, DHAP, 3PG, PEP, pyruvate). In LB (rich medium), however, glucose is scarce; E. coli catabolizes amino acids and peptides, so the pathway runs largely in the *gluconeogenic* direction to supply sugar-phosphates for cell wall, nucleotide, and cofactor biosynthesis. The lower segment (GAP ⇌ PEP) is shared by glycolysis and gluconeogenesis and is the bidirectional bottleneck.

**literature_support:** direct.
- Keio collection, Baba et al., 2006, *Mol. Syst. Biol.* (doi:10.1038/msb4100050): `gapA` could not be deleted on their standard medium; the authors explicitly attribute this to toxic intermediate accumulation in glycolysis null mutants.
- Nakahigashi et al., 2009, *Mol. Syst. Biol.* (doi:10.1038/msb.2009.65): of >70 central-carbon genes, only four (`fbaA`, `gapA`, `pgk`, and `eno`) are essential for growth on glucose as sole carbon source.
- Hidalgo et al., 2014, *Microb. Cell Fact.* (doi:10.1186/1475-2859-13-58): `tpiA` deletion causes only a partial growth defect in LB because methylglyoxal is produced but kept sub-toxic via the glyoxalase I/II bypass; glycerol growth is abolished.
- Shimada et al., 2011, *J. Bacteriol.* (PMC3021228): Cra activates `fbaB` under gluconeogenic conditions and represses `pfkA`, `pykF`.
- Py et al., 1996, *Nature* 381:169 (doi:10.1038/381169a0); Carpousis, 2007, *Annu. Rev. Microbiol.* (PMID:17447862): enolase is a structural component of the RNA degradosome.

**biological_plausibility:** strong. The subset `{fbaA, gapA, pgk, eno}` matches precisely the four central-carbon genes identified as essential even on glucose by Nakahigashi et al. (2009). These encode the only non-redundant enzymes on the GAP⇌3PG⇌PEP backbone; there is no isozyme or bypass. Genes with isozymes (`pfkA`/`pfkB`, `fbaA`/`fbaB`, `gpmA`/`gpmM`/`ytjC`, `pykF`/`pykA`) or with bypasses (`glk` bypassed by the PTS; `pgi` bypassed by zwf/pgl/gnd then tkt/tal; `tpiA` partially bypassed by the methylglyoxal shunt) tolerate single knockdown.

**specificity_to_rich_medium:** not-specific for the strong four, but the *interpretation* shifts. On glucose-minimal, `{fbaA, gapA, pgk, eno}` would be essential to deliver pyruvate and energy. In LB, flux is dominated by amino-acid catabolism and is net gluconeogenic; the same four genes remain essential because they operate in reverse to supply F1,6BP, F6P, and G6P for biosynthesis. `tpiA` being control-like is somewhat LB-favored: the rich medium supplies nucleotides, amino acids, and lipids, reducing the demand on DHAP→GAP conversion and buffering methylglyoxal toxicity. On glycerol minimal it would score strongly.

**hypothesis explaining variations:**
1. Essentiality reflects *enzymatic non-redundancy on the shared lower-glycolytic/gluconeogenic trunk*, not glycolytic flux per se. `gapA`, `pgk`, and `eno` have no functional isozyme under standard conditions; `fbaA` carries the bulk of aldolase activity because `fbaB` is Cra-activated and expressed mainly in gluconeogenic/stationary conditions (Shimada 2011). In exponential LB growth, the cell is already gluconeogenic but `fbaA` still dominates the protein pool, so `fbaA` KD collapses both directions.
2. `fbaA` and `fbaB` are *not parallel branches* — they catalyze the same reaction, but are isozymes with opposite regulation. `fbaB` KD is control-like because its basal expression is low; `fbaA` cannot be compensated on physiological timescales.
3. `tpiA` KD is control-like because DHAP can be drained through the methylglyoxal synthase (`mgsA`)/glyoxalase (`gloA`/`gloB`) shunt to pyruvate/lactate, avoiding accumulation. LB provides ribonucleotides and lipids, lowering DHAP→GAP demand; methylglyoxal stays below the ~0.6 mM toxicity threshold (Hidalgo 2014).
4. `glk` is dispensable because the PTS (`ptsHI`/`crr`/`ptsG`) phosphorylates glucose during uptake; LB has little free glucose anyway.
5. `pgi` is dispensable via the oxidative pentose-phosphate pathway (zwf→pgl→gnd) plus transketolase/transaldolase reconnecting to F6P/GAP.
6. `pfkA` KD: partially buffered by `pfkB` (≈10% of total PFK activity); the strongest expected phenotype is in glucose minimal, less in LB where upper glycolysis is bypassable.
7. `pykF`/`pykA` KD: PEP→pyruvate is bypassed by amino-acid catabolism (Ala, Ser, Gly, Thr, Cys) and by PEP carboxylase/malic enzyme cycling; in LB the pyruvate pool is supplied by proteolysis, so pyruvate kinase loss is nearly silent.
8. `eno` deserves a note: beyond catalysis, enolase is a structural component of the RNA degradosome (Py 1996, Carpousis 2007) and gates decay of `ptsG` mRNA (Morita et al. 2004, *Genes Dev.*); severe knockdown likely co-destabilizes RNA turnover, amplifying the phenotype beyond pure metabolic loss.
9. The pathway diagram implicitly places `eno` on the essential trunk; a weaker phenotype than `gapA/pgk` may reflect incomplete knockdown (`eno` is highly expressed, so residual protein persists longer after CRISPRi onset) rather than functional redundancy.

Direct answers to your specific questions:

- **Why `{fbaA, gapA, pgk, eno}` dominate:** these four are the non-redundant enzymes on the bidirectional trunk required for both glycolysis and gluconeogenesis in LB; their loss blocks both F1,6BP → PEP flux *and* PEP → F6P/G6P reflux needed for biosynthesis.
- **Are the others non-essential?** Yes, under LB, all others have isozymes (pfkA/pfkB, fbaA/fbaB, gpmA/M/ytjC, pykF/A), parallel bypasses (pgi→PPP, glk→PTS), or their substrate can be drained toxicity-free (DHAP via methylglyoxal shunt for `tpiA`). This matches Nakahigashi et al. (2009).
- **`tpiA` non-essential interpretation:** DHAP accumulation is handled by `mgsA`/`gloA`/`gloB` without exceeding the toxicity threshold in LB; biosynthetic demand for GAP is also covered by gluconeogenic reflux from 3-phospho-glycerate (via `pgk`+`gapA` running in reverse).
- **Do `tpiA` and `fbaA` correspond to different branches?** No. `fbaA` and `fbaB` are isozymes of the same reaction (not branches), and `tpiA` lies immediately downstream of `fbaA` on the DHAP arm. They are sequential, not parallel. The `tpiA`-tolerant phenotype simply reflects a drain (methylglyoxal shunt) and a parallel supply of GAP from upper glycolysis/PPP.
- **Would you expect `glk`, `pgi`, `pfkA`, `pfkB` to be essential?** No in LB. `glk` is redundant with the PTS; `pgi` is bypassable via PPP; `pfkA` single KD shows a mild defect on glucose and near-control on LB; `pfkB` alone is essentially silent.
- **`eno`, `pykF`, `pykA`?** `eno` should be essential (single copy, no isozyme, plus degradosome role) — if its phenotype is weaker than `gapA/pgk` in your screen, suspect incomplete depletion at the time of readout. `pykF` and `pykA` should be near-control in LB because pyruvate is generated directly from amino-acid catabolism and because the PEP-pyruvate node is buffered by Pck/PpsA/MaeA/MaeB cycling.

## follow-up (heatmap proposal)

**(i) Perturbations (rows):**
- Strong trunk: `fbaA`, `gapA`, `pgk`, `eno`
- Weaker / control-like glycolysis: `fbaB`, `tpiA`, `pfkA`, `pfkB`, `pgi`, `glk`, `gpmA`, `gpmM`, `pykF`, `pykA`
- Positive controls / orthogonal stressors: `mgsA` (methylglyoxal synthase), `gloA`, `gloB` (glyoxalase I/II), `rpoS` (general stress), a non-targeting sgRNA

**(ii) Genes whose expression to read (columns):**
- Cra regulon: `fbaB`, `ppsA`, `pckA`, `aceA`, `aceB`, `fbp`, `maeB` (gluconeogenic upregulation)
- CRP/catabolite readout: `ptsG`, `ptsH`, `crr`, `cyaA`
- Methylglyoxal / oxidative stress: `mgsA`, `gloA`, `gloB`, `yqhD`, `gldA`, plus general stress `soxS`, `oxyR`, `katG`, `dps`
- Stringent response / growth arrest: `relA`, `spoT`, `rmf`, `rpoS`, `hslO`
- Pentose phosphate bypass (relevant if `pgi`/upper-glycolysis is perturbed): `zwf`, `gnd`, `tktA`, `tktB`, `talA`, `talB`
- Degradosome-linked readout (for `eno` specifically): `rne`, `pnp`, `rhlB`, `ptsG` half-life proxy (steady-state `ptsG` mRNA)
- Own-gene transcripts (to verify CRISPRi efficacy at the mRNA level for each targeted gene)

**(iii) Expected patterns:**
- `{fbaA, gapA, pgk}` should co-cluster with a broad growth-arrest / stringent signature: up `rpoS`, `rmf`, `dps`; up `ppsA`, `pckA`, `fbp` (gluconeogenic attempt to refill upper glycolysis); down ribosomal/biosynthetic regulons. Methylglyoxal-related genes (`mgsA`, `gloA`) should rise specifically in `fbaA`/`tpiA` if DHAP backs up.
- `tpiA` should look like `fbaA`-lite on the methylglyoxal axis (up `gloA`/`gloB`, possibly up `yqhD`) but *without* the general stringent signature → this supports the "toxic metabolite but bypassable" hypothesis.
- `eno` should induce a *distinct* signature relative to `pgk`/`gapA`: in addition to the metabolic stress, `ptsG` mRNA should rise (SgrS-mediated decay is degradosome-dependent), and the composition or abundance of RNase-E-associated transcripts should shift. A unique `eno` cluster separating from `pgk`/`gapA` along a degradosome axis would be direct evidence of the moonlighting contribution.
- `fbaB`, `pfkB`, `glk`, `pykA` should cluster with the non-targeting control.
- `pfkA`, `pgi`, `pykF` should show intermediate, Cra/CRP-consistent compensations (up PPP genes for `pgi`; up `fbp`/`ppsA` for `pykF`) without the stringent signature.

The most discriminating single plot would be a two-axis summary: (stringent / growth-arrest score) vs. (methylglyoxal-response score), with a third color channel for (degradosome/ptsG-stability score). This should separate `eno` from the rest of the essential trunk and separate `tpiA` from the controls.

Your observations fit a parsimonious two-axis story; CRP and PPP actually fold into what you're already seeing, so you don't need to treat them as separate hypotheses.

### The simplified story

For `{fbaA, gapA, pgk, eno}` (core trunk), two — and only two — things are happening, and everything else follows:

**Axis 1. Cra turns on → gluconeogenesis attempt.** The block depletes F1,6BP, which releases Cra from its target promoters. Cra activates `ppsA`, `pckA`, `fbp`, `fbaB`, `aceA/B`, `maeB`. This is the "try to make sugars from amino-acid carbon" response.

**Axis 2. ppGpp rises → RpoS takes over → growth arrest.** Biosynthetic precursors run out, `spoT` (and/or `relA`) synthesizes ppGpp, translation slows, and `rpoS` replaces `σ70` for a large fraction of promoters. `rpoS` up is the single most informative marker.

Everything else you listed is a consequence of one of these two axes.

### CRP/catabolite, plainly

CRP-cAMP is the system that says "glucose is scarce → switch to alternative catabolism". Mechanistically: when the PTS is not actively pumping glucose, the EIIA component (`crr`) stays phosphorylated, which activates adenylate cyclase (`cyaA`) and raises cAMP. cAMP binds CRP, which then activates hundreds of catabolic operons (lactose, maltose, arabinose, TCA enzymes, etc.).

You see CRP targets *down* in the core trunk. That is not a third story; it is the **RpoS takeover**. As cells enter growth arrest, the σ70/CRP-driven growth program is globally displaced by the σS program. Many CRP-regulated catabolic operons lose their σ70 holoenzyme and decline. You can drop CRP as a separate axis — just note "CRP targets down" is a proxy for the same RpoS/growth-arrest signal.

### PPP "bypass" — same story, one level deeper

Your pattern `tktB ↑, talA ↑, tktA ↓, zwf ↓` is **not** a PPP bypass induced by a metabolic rerouting. It is the canonical σ70-to-σS isozyme swap within the PPP:

- `tktA`, `zwf` are σ70, exponential-phase isoforms → down with growth arrest.
- `tktB`, `talA` are σS/RpoS-dependent, stationary-phase isoforms → up when RpoS takes over.

So this group is also a readout of Axis 2, not an independent bypass signal. If an actual `pgi`-driven PPP flux reroute were occurring, you would see `zwf` *up* with `tktA` *up*; that is not what the data shows.

### Revised, minimal gene panel

Three functional groups covering both axes and the discriminators:

```python
gene_groups = [
    ("Cra / gluconeogenesis",  ["fbaB", "ppsA", "pckA", "fbp", "aceA", "aceB", "maeB"]),
    ("stringent / RpoS",       ["spoT", "relA", "rpoS", "rmf", "dps",
                                "tktB", "talA",          # RpoS-dependent PPP
                                "tktA", "zwf"]),          # σ70 PPP (expected down)
    ("discriminators",         ["mgsA", "gloA", "gloB",  # tpiA-specific
                                "ptsG", "rne", "pnp"]),  # eno-specific (degradosome)
]
```

### Interpretation in one sentence

Blocking the core glycolytic trunk in LB produces a stereotyped two-step response — Cra-driven gluconeogenic induction, then a global RpoS-mediated growth-arrest program — and both the CRP decline and the PPP "bypass" pattern are downstream consequences of the RpoS takeover rather than independent metabolic signals. The only KDs expected to deviate from this two-axis pattern are `tpiA` (methylglyoxal signature without the stringent arm) and `eno` (add a degradosome signature on top of the two axes).

#### Adenine ribonucleotide biosynthesis

In [ ]:
module_id = "eco_M00049"

perturbations = [
    "adk", "purA", "purB", "ndk", "nontargeting"
]
gene_names = [
    "uspA",  # energy/stress-responsive; senses ATP directly as cofactor
    "rpoS",  # it implies a SHIFT in transcriptional priority toward stress-protective genes
    "relA",  # RelA synthesizes ppGpp → triggers stringent response (slows down translation)
    "spoT",  # bifunctional ppGpp synthetase/hydrolase; primary responder to non-amino-acid stresses
    "alaS",  # aminoacyl-tRNA synthetase for alanine; proxy for translation fidelity / amino acid availability
    "amn",   # AMP nucleosidase; degrades AMP → adenine + R5P; marker of AMP accumulation
    "purR"   # transcriptional repressor of de novo purine synthesis; de-repressed when purines are limiting
]

plot_expression_data(adata, perturbations, gene_names)

# follow up observations
# uspA, rpoS are upregulated in adk
# relA, spotT seem downregulated in adk compared to the other perturbations
# amn, purR are not expressed at high levels

In [ ]:
adata_sel = adata[adata.obs["target"].isin(perturbations)].copy()
sc.pp.pca(adata_sel, n_comps=10)
sc.pp.neighbors(adata_sel, use_rep="X_pca")
sc.tl.umap(adata_sel)
sc.pl.umap(adata_sel, color=["target"], )

The distinct phenotype of *adk* knockdown compared to other pathway genes is due to its absolute essentiality for cellular energy homeostasis. In rich medium, *purA* and *purB* defects can be bypassed by salvaging adenine or adenosine to AMP. *ndk* has redundant enzymes (e.g., glycolytic kinases) that can produce ATP from ADP. However, *adk* is the sole enzyme that can produce ADP from AMP or interconvert 2 ADP $\rightleftharpoons$ ATP + AMP. Therefore, reducing Adk activity immediately disrupts the adenylate pool balance (energy charge), leading to severe growth arrest, a phenotype supported by the use of temperature-sensitive *adk* mutants in literature due to its non-deletable nature.

In [ ]:
fitness_data.loc[lambda x: x["gene"].isin(perturbations)]

#### NAD biosynthesis

In [ ]:
module_id = "eco_M00049"

perturbations = [
    "nadB","nadA", "nadC", "nadD", "nadE", 
    "nontargeting"
]
gene_names = [
    "uspA",  # energy/stress-responsive; senses ATP directly as cofactor
    "rpoS",  # it implies a SHIFT in transcriptional priority toward stress-protective genes
    "relA",  # RelA synthesizes ppGpp → triggers stringent response (slows down translation)
    "spoT",  # bifunctional ppGpp synthetase/hydrolase; primary responder to non-amino-acid stresses
    "alaS",  # aminoacyl-tRNA synthetase for alanine; proxy for translation fidelity / amino acid availability
]

plot_expression_data(adata, perturbations, gene_names)

# follow up observations
# clear upregualtion of rpoS in nadC
# Quinolinic acid

In [ ]:
compare_pairs(adata, "nadC", "nadD")

In [ ]:
gene_names = [
    "rplJ", "rplE", "rplD", "rplU",
    "rpsE", "rpsS", "rpsD", "rpsM",
    "rpoA",
    "hdeA", "hdeD", "gadC", "rpoS", "mscL"
]

plot_expression_data(adata, perturbations, gene_names)

In [ ]:
adata_sel = adata[adata.obs["target"].isin(perturbations)].copy()
sc.pp.pca(adata_sel, n_comps=10)
sc.pp.neighbors(adata_sel, use_rep="X_pca")
sc.tl.umap(adata_sel)
sc.pl.umap(adata_sel, color=["target"], )

In [ ]:
adata_sel.obs.groupby([""])

In [ ]:
fitness_data.loc[lambda x: x["gene"].isin(perturbations)]

discontinuity at nadC also observed in the fitness readouts, but with a different interpretation

#### pantothenate => CoA

In [ ]:
module_id = "eco_M00049"

perturbations = [
    "coaA",
    "dfp",
    "coaD",
    "coaE",
]
gene_names = [
    "uspA",  # energy/stress-responsive; senses ATP directly as cofactor
    "rpoS",  # it implies a SHIFT in transcriptional priority toward stress-protective genes
    "relA",  # RelA synthesizes ppGpp → triggers stringent response (slows down translation)
    "spoT",  # bifunctional ppGpp synthetase/hydrolase; primary responder to non-amino-acid stresses
    "alaS",  # aminoacyl-tRNA synthetase for alanine; proxy for translation fidelity / amino acid availability
]

plot_expression_data(adata, perturbations, gene_names)

In [ ]:
compare_pairs(adata, "coaA", "dfp")

In [ ]:
gene_names = [
    "rpoS",     # stress severity
    "raiA",     # ribosome hibernation / growth arrest depth
    "hdeA",     # acid/stationary stress
    "pta",      # CoA-flux proxy
    "coaA",     # does it transcriptionally compensate when downstream is blocked?
    "rpsJ",     # translation activity (ribosomal proxy)
    "yeaG",     # energy/carbon stress
]

plot_expression_data(adata, perturbations, gene_names)

In [ ]:
fitness_data.loc[lambda x: x["gene"].isin(perturbations)]

#### riboflavin (vitamin B2) biosynthesis

Riboflavin is the precursor to two essential electron-carrier cofactors, FMN and FAD. Both contain an isoalloxazine ring system that can reversibly accept and donate electrons (redox cycling between oxidized and reduced forms). They are chemically distinct from NAD — while NAD carries hydride (2-electron transfer), flavins can handle both 1- and 2-electron transfers, making them uniquely versatile in redox biochemistry.

In [ ]:
module_id = "eco_M00049"

perturbations = [
    "ribA",
    "ribD",
    "ribD",
    "ybjI",
    "ribB",
    "ribE",
    "ribC",
    "ribF",
]
gene_names = [
    "uspA",  # energy/stress-responsive; senses ATP directly as cofactor
    "rpoS",  # it implies a SHIFT in transcriptional priority toward stress-protective genes
    "relA",  # RelA synthesizes ppGpp → triggers stringent response (slows down translation)
    "spoT",  # bifunctional ppGpp synthetase/hydrolase; primary responder to non-amino-acid stresses
    "alaS",  # aminoacyl-tRNA synthetase for alanine; proxy for translation fidelity / amino acid availability
]

plot_expression_data(adata, perturbations, gene_names)

In [ ]:
compare_pairs(adata, "ribF", "ribC")

In [ ]:
fitness_data.loc[lambda x: x["gene"].isin(perturbations)]

why ribB so small / so control like?

Michaelis–Menten equation

$$
v = \frac{dp}{dt} = \frac{V_{max} \times [S]}{K_m + [S]}
$$

- $v$: flux
- [S]: substrate concentration
- $p$ product
- $V_{max}$: limiting rate (max flux)
- $K_m$: Michaelis constant: concentration $[S]$ at which the reaction rate is half of $V_max$

interpretation: kinetics, definitely need to read more on that

To answer this directly, we must look at how enzymes are kinetically optimized by evolution. The difference between how intermediate and terminal enzymes operate comes down to **enzyme kinetics (Michaelis-Menten dynamics)** and **cellular protein economy**. 

When we say an enzyme operates "at its maximum rate," we mean it is kinetically **saturated**—operating near its $V_{max}$, where an increase in substrate concentration ($[S]$) does not significantly increase the reaction velocity ($v$). When we say an enzyme operates below its maximum rate, it means it is operating in a linear regime where an increase in $[S]$ directly increases $v$.

Here is the exact biochemical rationale for why intermediate enzymes operate below maximum capacity, while terminal enzymes operate closer to it:

**1. The Kinetic Operating Regime ($[S]$ relative to $K_m$)**
Every enzyme has a Michaelis constant ($K_m$), which is the substrate concentration at which the enzyme operates at half its maximum velocity. 

*   **Intermediate enzymes operate in the first-order kinetic regime ($[S] < K_m$):** 
    Evolution optimizes intermediate enzymes so that their typical cellular substrate concentration is lower than their $K_m$. In this state, the reaction velocity is directly proportional to the substrate concentration ($v \approx \frac{V_{max}}{K_m}[S]$). 
    *Why?* This provides intrinsic mass-action stabilization. If the flux from the previous step increases, the intermediate substrate concentration rises slightly, and the enzyme's velocity immediately scales up to process it. The enzyme has excess kinetic capacity built-in.
*   **Terminal enzymes operate closer to the zero-order kinetic regime ($[S] \ge K_m$):**
    Terminal enzymes are often optimized so that their baseline substrate concentration saturates or nearly saturates the enzyme. In this state, velocity approaches a constant ($v \approx V_{max}$).
    *Why?* The cell needs a tightly controlled, stable output of the final product (like FMN/FAD), regardless of upstream fluctuations. A saturated terminal enzyme produces product at a constant, steady rate determined only by the amount of enzyme present, decoupling the final production rate from transient spikes in upstream intermediate concentrations.

**Prevention of Intermediate Toxicity and Osmotic Load**
Intermediates in biosynthetic pathways are biologically useless. Furthermore, they are often chemically reactive (prone to spontaneous degradation) or, if allowed to accumulate in large quantities, create a severe osmotic burden on the cell. 

To prevent intermediate accumulation, the cell over-expresses intermediate enzymes relative to the necessary flux. Because there is an excess of intermediate enzyme molecules compared to the number of substrate molecules, the intermediate enzymes spend a significant fraction of their time idle (operating far below their theoretical $V_{max}$). As soon as a substrate molecule is generated, it is immediately bound and converted.

**Thermodynamics and Irreversibility**
Intermediate reactions in sequential pathways are frequently reversible and operate near thermodynamic equilibrium ($\Delta G \approx 0$). Because they are near equilibrium, the net forward rate is dictated purely by the ratio of substrates to products. These enzymes must maintain high specific activity (excess capacity) simply to facilitate rapid equilibrium.

Terminal reactions, however, are typically the heavily exergonic ($\Delta G \ll 0$), committed steps that pull the entire pathway forward. For example, RibF utilizes ATP and CTP to phosphorylate and adenylylate its substrates. Because this consumes high-energy cellular currency, the cell tightly restricts the total number of these terminal enzyme molecules to prevent runaway consumption of ATP/CTP. Restricting the absolute number of terminal enzymes forces the existing ones to operate continuously at near-maximum turnover to meet cellular demand.

### Summary
Intermediate enzymes are highly abundant relative to their substrates and operate in a concentration-dependent manner ($[S] < K_m$) to instantly clear useless or toxic intermediates. Therefore, if you knock them down, the accumulation of their substrate pushes the remaining enzymes closer to their unused $V_{max}$, recovering the flux.

Terminal enzymes are restricted in number to conserve ATP and strictly regulate final product output. They operate near saturation ($[S] \ge K_m$). Because they are already operating near their maximum catalytic capacity, any knockdown immediately reduces the absolute flux of the final product, as the remaining enzymes have no unused kinetic capacity to tap into.

#### isoprenoid biosynthesis, bacteria

synthesize geranyl diphosphate (GPP, C10), farnesyl diphosphate (FPP, C15), and geranylgeranyl diphosphate (GGPP, C20). These are essential precursors for the synthesis of quinones (ubiquinone, menaquinone), hemes, and bactoprenol (required for peptidoglycan synthesis)

In [ ]:
module_id = "eco_M00049"

perturbations = [
    "idi",
    "ispA",
]
gene_names = [
    "uspA",  # energy/stress-responsive; senses ATP directly as cofactor
    "rpoS",  # it implies a SHIFT in transcriptional priority toward stress-protective genes
    "relA",  # RelA synthesizes ppGpp → triggers stringent response (slows down translation)
    "spoT",  # bifunctional ppGpp synthetase/hydrolase; primary responder to non-amino-acid stresses
    "alaS",  # aminoacyl-tRNA synthetase for alanine; proxy for translation fidelity / amino acid availability
]

compare_pairs(adata, "ispA", "idi")

In [ ]:
fitness_data.loc[lambda x: x["gene"].isin(perturbations)]


**interpretation:** There are two pathways that can synthetize DMAPP, the isoprenoid precursor.
The first one, the mevalonate pathway, relies on idi to convert IPP into DMPP.
However, in E. coli, the MEP pathway also allows for DMAPP synthesis.